# 分散 Self-Play — Colab worker

試合を生成してシャード(経験データ)を1ファイル出すだけのノートブック。学習はしない。

**GPU は不要**(むしろ意味がない)。ランタイムは CPU のままでよい。

手順: 1〜3 を上から実行 → 4 で Linux 動作確認 → 5 で本番収集 → 6 でダウンロード。


## 1. 環境確認


In [ ]:
!python -V
!nproc
!free -g | head -2
# numpy だけあればよい(worker は torch を使わない)
import numpy; print('numpy', numpy.__version__)


## 2. ファイルをアップロード

`ptcg_repo.zip`(コード一式)と `run_<id>_v<N>.zip`(設定とモデル)の2つを選ぶ。
リポジトリ側は毎回同じなので、2回目以降は run の zip だけでよい。


In [ ]:
from google.colab import files
up = files.upload()
print(list(up))


In [ ]:
import os, zipfile, glob

if os.path.exists('ptcg_repo.zip'):
    os.makedirs('/content/ptcg', exist_ok=True)
    zipfile.ZipFile('ptcg_repo.zip').extractall('/content/ptcg')
for z in glob.glob('run_*.zip'):
    zipfile.ZipFile(z).extractall('/content')   # -> /content/run/

!ls /content/run /content/run/models
!cat /content/run/run.json


## 3. ゲームエンジンが動くか

`libcg.so` が読めて対戦が回ることの確認。ここが通らなければ先に進めない。


In [ ]:
%cd /content/ptcg/kaggle_replays/rl
!python test_rollout.py


## 4. 並列処理の起動方式を確定させる

**このノートブックの一番の目的。** spawn と fork の両方を実際に走らせて、
Linux でどちらが使えるかを見る。結果を控えておくこと。


In [ ]:
# Linux での並列処理の起動方式を確かめる。
#
# 手元(Windows)は spawn が既定なので検証済みだが、Linux の既定は fork で、
# 親が読み込み済みの cg エンジン(ネイティブライブラリ)を子が引き継ぐ。ここで
# 両方を実際に走らせて、どちらが使えるかを確定させる。
import subprocess, sys, os

os.chdir('/content/ptcg/kaggle_replays/rl/distributed')

# 診断用に試合数の少ない run を作る(本番の run とは別物)
subprocess.run([sys.executable, 'init_run.py', '--run-id', 'diag',
                '--run-dir', '/content/diagrun', '--workers', 'w0',
                '--games-per-worker', '8'], check=True)

for method in ('spawn', 'fork'):
    print(f'\n===== start-method = {method} =====', flush=True)
    r = subprocess.run(
        [sys.executable, 'worker.py', '--run-dir', '/content/diagrun',
         '--worker-id', 'w0', '--workers', '2', '--start-method', method,
         '--out', f'/tmp/diag_{method}.npz'],
        capture_output=True, text=True, timeout=1800)
    print(r.stdout[-2000:])
    if r.returncode != 0:
        print('--- stderr ---'); print(r.stderr[-2000:])
    print(f'>>> {method}: ' + ('OK' if r.returncode == 0 else f'NG (exit {r.returncode})'))


## 5. 本番の収集

4 で OK だった方式を `--start-method` に指定する(既定は spawn)。
`--workers` はこのマシンのコア数(1 の `nproc` の値)。


In [ ]:
%cd /content/ptcg/kaggle_replays/rl/distributed
!python worker.py --run-dir /content/run --worker-id colab --workers 2 --start-method spawn


## 6. シャードを回収

落としたファイルを学習PCの `<run_dir>/shards/v<N>/` へ置く。


In [ ]:
import glob
from google.colab import files
for p in glob.glob('/content/run/shards/v*/colab.npz'):
    print(p)
    files.download(p)
